In [ ]:
import chromadb
from chromadb.config import Settings

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Create persistent client
# ─────────────────────────────────────────────────────────────────────────────

def get_chroma_client(persist_path: str = "./chroma_db") -> chromadb.PersistentClient:
    """
    Creates (or reconnects to) a persistent ChromaDB client.
    Data is saved to disk at persist_path — survives notebook restarts.
    """
    client = chromadb.PersistentClient(path=persist_path)
    print(f"✅ ChromaDB client connected at '{persist_path}'")
    return client


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Get or create collection (cosine locked in here)
# ─────────────────────────────────────────────────────────────────────────────

def get_or_create_collection(
    client: chromadb.PersistentClient,
    collection_name: str = "rag_papers",
) -> chromadb.Collection:
    """
    Returns existing collection or creates a new one.

    Key decisions locked in at creation:
      - hnsw:space = cosine  →  correct for OpenAI text embeddings
      - get_or_create        →  safe to re-run notebook cells without errors
    """
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},   # 🔴 permanent — cannot change later
    )
    print(f"✅ Collection '{collection_name}' ready ({collection.count()} chunks stored)")
    return collection


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Add chunks to collection
# ─────────────────────────────────────────────────────────────────────────────

def add_chunks_to_collection(
    collection: chromadb.Collection,
    chunks: list[dict],
    source_name: str,
) -> None:
    """
    Adds enriched chunks (from add_contextual_retrieval) to ChromaDB.

    Expects each chunk to have:
        - contextualized_text  (from contextual enrichment step)
        - text                 (raw chunk text)
        - context              (Claude-generated context prefix)
        - page_start, page_end (from semantic chunker)
        - chunk_id             (from semantic chunker)

    Uses upsert so re-running won't create duplicates.
    """
    if not chunks:
        print(f"  ⚠️  No chunks to add for '{source_name}'")
        return

    ids         = []
    embeddings  = []
    documents   = []
    metadatas   = []

    for chunk in chunks:
        # Unique ID: source + chunk_id prevents collisions across multiple PDFs
        chunk_id = f"{source_name}_chunk_{chunk['chunk_id']}"

        ids.append(chunk_id)
        embeddings.append(chunk["embedding"])           # pre-computed OpenAI vector
        documents.append(chunk["contextualized_text"])  # what gets retrieved
        metadatas.append({
            "source":      source_name,
            "chunk_id":    chunk["chunk_id"],
            "page_start":  chunk["page_start"],
            "page_end":    chunk["page_end"],
            "sentences":   chunk["sentences"],
            "raw_text":    chunk["text"],               # handy to have original alongside contextualised
            "has_context": bool(chunk.get("context")),
        })

    # upsert = insert if new, update if ID already exists
    # safe to re-run without creating duplicates
    collection.upsert(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
    )
    print(f"  ✅ {len(chunks)} chunks upserted for '{source_name}'")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Query the collection
# ─────────────────────────────────────────────────────────────────────────────

def query_collection(
    collection: chromadb.Collection,
    query_embedding: list[float],
    n_results: int = 20,            # blog recommends 20 as most performant
    source_filter: str = None,      # optional: restrict to one PDF
) -> list[dict]:
    """
    Queries ChromaDB and returns top-n chunks.

    Args:
        query_embedding: pre-computed embedding of the user's query (OpenAI)
        n_results:       number of chunks to return (blog recommends 20)
        source_filter:   optional — only search within a specific source PDF

    Returns a clean list of dicts with text, metadata, and similarity distance.
    """
    # Optional metadata filter — freely add/change anytime, no rebuild needed
    where = {"source": source_filter} if source_filter else None

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where,                          # None = search entire collection
        include=["documents", "metadatas", "distances"],
    )

    # Flatten ChromaDB's nested response into a clean list
    output = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        output.append({
            "text":       doc,
            "source":     meta["source"],
            "page_start": meta["page_start"],
            "page_end":   meta["page_end"],
            "raw_text":   meta["raw_text"],
            "distance":   round(dist, 4),       # cosine distance: 0 = identical, 2 = opposite
            "similarity": round(1 - dist, 4),   # convert to similarity score for readability
        })

    return output


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Utility helpers
# ─────────────────────────────────────────────────────────────────────────────

def collection_stats(collection: chromadb.Collection) -> None:
    """Prints a quick summary of what's stored in the collection."""
    count = collection.count()
    print(f"\n📊 Collection: '{collection.name}'")
    print(f"   Total chunks stored : {count}")
    print(f"   Distance metric     : {collection.metadata.get('hnsw:space', 'l2')}")

    if count > 0:
        # Sample first result to show available metadata fields
        sample = collection.peek(limit=1)
        print(f"   Metadata fields     : {list(sample['metadatas'][0].keys())}")


def delete_collection(
    client: chromadb.PersistentClient,
    collection_name: str,
) -> None:
    """
    Permanently deletes a collection and all its data.
    Use when you need to rebuild with different settings (e.g. wrong distance metric).
    """
    client.delete_collection(name=collection_name)
    print(f"🗑️  Collection '{collection_name}' deleted")